# COMP4330 Final Runner

This notebook reproduces the frozen final method: **Selective Tiled Inference** on VGG16-SSD300.

The project started from a 77.66% overall / 10.78% small-object AP baseline, tested a controlled 6k `conv7 -> conv4_3` fusion comparison, diagnosed and repaired its gate behavior, and screened two alternative fusion structures. Those experiments did not produce a reliable gain, so the final method enlarges small objects at inference time without changing the detector weights. The full investigation, including rejected results and their likely causes, is in [`reports/experiment_history.md`](../reports/experiment_history.md).

Expected full VOC2007 test result:

- overall mAP at IoU 0.5: **78.33%** (77.66% baseline)
- small AP at IoU 0.5: **30.65%** (10.78% baseline)
- small recall at IoU 0.5: **90.99%** (77.05% baseline)
- FPS: **5.45** (19.06 baseline)

Use a T4 GPU runtime. The quick cells run tests, generate the result figure with Matplotlib, and run a five-image smoke check. The complete evaluation is opt-in because it takes about 20 minutes.

Repository: https://github.com/itsvvy/comp4330-ssd-fusion

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

REPO_URL = "https://github.com/itsvvy/comp4330-ssd-fusion.git"
PROJECT_DIR = Path("/content/comp4330-ssd-fusion")
ASSET_ROOT = Path("/content/assets")
DATA_ROOT = Path("/content/data/VOCdevkit")
RESULT_ROOT = Path("/content/results")

if not PROJECT_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", "main", "--depth", "1", REPO_URL, str(PROJECT_DIR)],
        check=True,
    )
os.chdir(PROJECT_DIR)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print("project:", PROJECT_DIR)
print("commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
!python -m pip install -q -r requirements.txt

In [ ]:
import platform
import torch

print("python:", platform.python_version())
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda, torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram_gib:", torch.cuda.get_device_properties(0).total_memory / 2**30)
print("disk free GiB:", shutil.disk_usage("/content").free / 2**30)

In [ ]:
# Assets are written to ephemeral /content storage. Re-run this cell after
# a runtime reset. The script verifies all downloads against SHA256SUMS.json.
!bash scripts/download_assets.sh "{ASSET_ROOT}" "{DATA_ROOT}"

In [ ]:
!python -m pytest -q

In [ ]:
# Standard Python plotting from the frozen experiment JSON.
!python scripts/plot_results.py \
  --history reports/experiment_history.json \
  --output-dir reports/figures
from IPython.display import Image, display
display(Image(filename="reports/figures/experiment_overview.png"))

In [ ]:
SMOKE_DIR = RESULT_ROOT / "smoke_tiled"
!python tools/tiled_eval.py \
  --trained_model "{ASSET_ROOT}/weights/ssd300_mAP_77.43_v2.pth" \
  --voc_root "{DATA_ROOT}" \
  --save_folder "{SMOKE_DIR}" \
  --variant baseline \
  --tile_grid 2 \
  --overlap 0.2 \
  --include_full \
  --tile_max_area 1024 \
  --limit 5
!python tools/size_stratified_eval.py \
  --detections "{SMOKE_DIR}/detections.pkl" \
  --voc-root "{DATA_ROOT}" \
  --output "{SMOKE_DIR}/size_metrics.json" \
  --limit 5

## Full reproduction

Run the following cell only when a complete 4,952-image result is needed. The parameters are frozen and must not be tuned on the test set.

In [ ]:
RUN_FULL = False
FULL_DIR = RESULT_ROOT / "full_tiled_small"

if RUN_FULL:
    subprocess.run(
        ["python", "tools/tiled_eval.py",
         "--trained_model", str(ASSET_ROOT / "weights/ssd300_mAP_77.43_v2.pth"),
         "--voc_root", str(DATA_ROOT),
         "--save_folder", str(FULL_DIR),
         "--variant", "baseline",
         "--tile_grid", "2",
         "--overlap", "0.2",
         "--include_full",
         "--tile_max_area", "1024"],
        check=True,
    )
    subprocess.run(
        ["python", "tools/size_stratified_eval.py",
         "--detections", str(FULL_DIR / "detections.pkl"),
         "--voc-root", str(DATA_ROOT),
         "--output", str(FULL_DIR / "size_metrics.json")],
        check=True,
    )
else:
    print("Set RUN_FULL = True to reproduce the complete VOC2007 evaluation.")

## Artifacts and cleanup

The full run writes `detections.pkl`, `metrics.json`, and `size_metrics.json` under `/content/results/full_tiled_small`.

The frozen report is `reports/tiled_inference_results.md`; exact values are in `reports/tiled_inference_results.json`. Fusion experiments are retained on independent Git branches and are not part of the default workflow.

After evaluation, disconnect the Colab runtime from **Runtime > Disconnect and delete runtime**.